<a href="https://colab.research.google.com/github/hari96-afk/MSAI-631-A02-HariRamachandran/blob/main/Recommendation_System_using_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# ==============================
# LOAD DATA
# ==============================
movies = pd.read_csv("movies.csv")

# Clean data
movies['title'] = movies['title'].fillna('')
movies['genres'] = movies['genres'].fillna('')

# Normalize titles for better matching
movies['clean_title'] = movies['title'].str.lower().str.strip()

# Combine features
movies['features'] = movies['title'] + " " + movies['genres']

# ==============================
# TF-IDF MODEL
# ==============================
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['features'])

# RAM SAFE MODEL
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(tfidf_matrix)

# ==============================
# SMART SEARCH FUNCTION
# ==============================
def find_movie(title):
    title = title.lower().strip()

    # Exact match
    matches = movies[movies['clean_title'] == title]

    if len(matches) > 0:
        return matches.index[0]

    # Partial match
    matches = movies[movies['clean_title'].str.contains(title, na=False)]

    if len(matches) > 0:
        return matches.index[0]

    return None

# ==============================
# RECOMMENDATION FUNCTION
# ==============================
def recommend(movie_name, n=5):

    idx = find_movie(movie_name)

    if idx is None:
        return ["Movie not found in dataset. Try a different name."]

    distances, indices = model.kneighbors(tfidf_matrix[idx], n_neighbors=n+1)

    results = []

    for i in range(1, len(indices[0])):  # skip itself
        results.append(movies['title'].iloc[indices[0][i]])

    return results

# ==============================
# USER INTERFACE
# ==============================
print("\n===================================")
print(" MOVIE RECOMMENDER SYSTEM ")
print("===================================\n")

try:
    movie_name = input("Enter Movie Name: ")

    results = recommend(movie_name)

    print("\nTop 5 Recommendations:\n")

    for i, movie in enumerate(results, 1):
        print(f"{i}. {movie}")

except Exception as e:
    print("Error occurred:", e)
    print("Please try again.")


 MOVIE RECOMMENDER SYSTEM 

Enter Movie Name: afdsdgf

Top 5 Recommendations:

1. Movie not found in dataset. Try a different name.
